# 07 – Export for Tableau

Save small, summarised CSV files in `dashboard/data/` for the Tableau dashboard.

In [3]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

conn = sqlite3.connect("../data/processed/procurement.db")
OUT = Path("../dashboard/data")
OUT.mkdir(parents=True, exist_ok=True)

def q(sql):
    return pd.read_sql(sql, conn)

def save(df, name):
    df.to_csv(OUT / name, index=False)
    print(f"{name}: {len(df):,} rows")

# 1. Monthly spend by group
save(q("""
    SELECT report_month || '-01' AS month, spend_group, COUNT(*) AS payments, ROUND(SUM(amount), 2) AS total
    FROM payments GROUP BY report_month, spend_group ORDER BY report_month, spend_group
"""), "monthly_spend.csv")

# 2. Top 25 Procurement suppliers
save(q("""
    SELECT supplier_clean AS supplier, COUNT(*) AS payments, ROUND(SUM(amount), 2) AS total,
           ROUND(100.0 * SUM(amount) / (SELECT SUM(amount) FROM payments
                 WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'), 2) AS share_pct
    FROM payments WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'
    GROUP BY supplier_clean ORDER BY total DESC LIMIT 25
"""), "top_suppliers.csv")

# 3. Procurement spend by type (top 20)
save(q("""
    SELECT expense_type, COUNT(*) AS payments, ROUND(SUM(amount), 2) AS total
    FROM payments WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'
    GROUP BY expense_type ORDER BY total DESC LIMIT 20
"""), "spend_by_type.csv")

monthly_spend.csv: 72 rows
top_suppliers.csv: 25 rows
spend_by_type.csv: 20 rows


In [4]:
# 4. Supplier flag summary
save(q("""
    SELECT supplier_clean AS supplier, COUNT(*) AS payments, SUM(risk_score > 0) AS flagged,
           ROUND(100.0 * SUM(risk_score > 0) / COUNT(*), 1) AS flagged_pct,
           ROUND(SUM(CASE WHEN risk_score > 0 THEN amount END), 2) AS flagged_value
    FROM payment_flags GROUP BY supplier_clean
    HAVING SUM(risk_score > 0) >= 5 ORDER BY flagged_value DESC LIMIT 25
"""), "supplier_flags.csv")

# 5. Flagged payments
save(q("""
    SELECT payment_id, supplier_clean AS supplier, pay_date, ROUND(amount, 2) AS amount,
           expense_type, service_area, flag_duplicate_pair, flag_round_large,
           flag_weekend, flag_date_mismatch, risk_score
    FROM payment_flags WHERE risk_score > 0 ORDER BY amount DESC
"""), "flagged_payments.csv")

# 6. Benford's Law
d = q("""SELECT supplier_clean, amount FROM payments
         WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED' AND amount >= 10""")

def first_digit_share(a):
    fd = (a / 10 ** np.floor(np.log10(a))).astype(int)
    return fd.value_counts(normalize=True).reindex(range(1, 10), fill_value=0).sort_index()

all_share = first_digit_share(d["amount"])
uniq_share = first_digit_share(d.drop_duplicates(["supplier_clean", "amount"])["amount"])
save(pd.DataFrame({
    "first_digit": range(1, 10),
    "expected_pct": [round(np.log10(1 + 1 / k) * 100, 1) for k in range(1, 10)],
    "all_payments_pct": (all_share.values * 100).round(1),
    "unique_amounts_pct": (uniq_share.values * 100).round(1),
}), "benford.csv")

supplier_flags.csv: 25 rows
flagged_payments.csv: 1,093 rows
benford.csv: 9 rows
